# MATWM + RND (Random Network Distillation) Intrinsic Rewards

## 目的

非中央集権型MARL（Multi-Agent Reinforcement Learning）において、
**RND (Random Network Distillation) による内発的報酬**を導入し、探索効率を改善する。

- **環境**: PettingZoo `simple_tag_v3`（3 追手 vs 1 逃走者）
- **ベースモデル**: MATWM (Multi-Agent Transformer World Model)
- **探索手法**: RND (Burda et al., 2018)

---

## RND (Random Network Distillation) とは

RND は観測の「新規性」を測定する内発的報酬手法です。

### 仕組み

1. **Target Network**: ランダムに初期化され、固定されたニューラルネットワーク
   - 入力: 観測 `s`
   - 出力: 固定された特徴ベクトル `f_target(s)`

2. **Predictor Network**: 訓練可能なニューラルネットワーク
   - 入力: 観測 `s`
   - 出力: 予測された特徴ベクトル `f_pred(s)`
   - 目的: Target の出力を模倣するように学習

3. **内発的報酬**: 予測誤差 = 新規性
   ```
   r_intrinsic = || f_target(s) - f_pred(s) ||^2
   ```

### なぜ機能するか

- **訪問済みの状態**: Predictor が何度も見ているため、予測精度が高い → 低い報酬
- **未訪問の状態**: Predictor が見たことがないため、予測誤差が大きい → 高い報酬

### 利点

- モデルフリー：環境ダイナミクスのモデル化が不要
- シンプル：予測誤差を最小化するだけ
- スケーラブル：高次元観測空間でも機能

---

## なぜ探索が必要か（非中央集権型MARLにおけるモチベーション）

### 1. 部分観測性と信用割当の困難
各エージェントは局所的な観測しか得られず、環境報酬の変動が自分の行動の結果なのか、
他エージェントの行動の結果なのか区別できない。

### 2. 探索の局所化
中央集権型では全体の状態空間を見渡して探索を誘導できるが、
非中央集権型では「全員が同じ局所最適に陥る」相関探索問題が発生する。
RND は各エージェントに個別の探索動機を与え、探索行動を多様化させる。

### 3. 報酬の希薄性
追手チームが獲物を捕まえるまで有意な報酬が得られない。
RND による内発的報酬は、観測空間の新規性に基づく密な報酬シグナルを提供する。

## 1. セットアップ

In [ ]:
# Add project root to Python path for importing modules
import sys
from pathlib import Path

# Get project root (2 levels up from current notebook location)
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

print(f"Project root added to path: {project_root}")

In [ ]:
# 必要パッケージのインストール
%pip install pygame
%pip install --no-deps pettingzoo
%pip install numpy gymnasium supersuit
%pip install torchinfo

import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 環境の作成と仕様確認
from pettingzoo.mpe import simple_tag_v3

def make_env(max_cycles=25, seed=None):
    env = simple_tag_v3.parallel_env(
        num_good=1,
        num_adversaries=3,
        num_obstacles=2,
        max_cycles=max_cycles,
        continuous_actions=False,
        render_mode=None,
    )
    if seed is not None:
        env.reset(seed=seed)
    return env

env = make_env(seed=0)
obs, info = env.reset(seed=0)

print('=== Environment Specifications ===')
print('Agents:', env.agents)
print('\nObservation shapes:')
for agent in env.agents:
    print(f'  {agent}: {obs[agent].shape}')
print('\nAction spaces:')
for agent in env.agents:
    print(f'  {agent}: {env.action_space(agent)}')
env.close()

## 2. MATWM 実装の読み込み

In [ ]:
from matwm_implementation import MATWMConfig, pad_observation
from matwm_utils import (
    initialize_matwm_weights, init_weights,
    save_full_checkpoint, load_full_checkpoint,
    plot_training_progress,
    inspect_matwm_architecture,
    print_gpu_info, setup_matwm_training
)
from matwm_agent import MATWMAgent

# 設定
config = MATWMConfig(
    total_steps=50000,   # 論文再現時は50000
    warmup_steps=1000,   # 論文再現時は1000
    log_interval=100,
    save_interval=1000,
    use_gamma_progress=False  # RNDではWorld Model関連の好奇心を使わない
)

print('=== MATWM Configuration ===')
print(f'  Total steps: {config.total_steps}')
print(f'  Warmup steps: {config.warmup_steps}')
print(f'  Latent dim: {config.latent_dim}x{config.num_classes}')
print(f'  Imagination horizon: {config.imagination_horizon}')
print(f'  Max observation dim: {config.max_obs_dim} (zero-padding)')

In [ ]:
# 観測パディングのテスト
print('=== Observation Padding Test ===')
test_obs_14 = np.random.randn(14)
test_obs_16 = np.random.randn(16)

padded_14 = pad_observation(test_obs_14, config.max_obs_dim)
padded_16 = pad_observation(test_obs_16, config.max_obs_dim)

print(f'14 dim -> {padded_14.shape}, last 2 = {padded_14[-2:]} (should be 0)')
print(f'16 dim -> {padded_16.shape}, unchanged = {np.allclose(test_obs_16, padded_16)}')
print('✓ Zero-padding OK')

In [ ]:
# GPU環境とアーキテクチャ確認
gpu_info = print_gpu_info()
setup_info = setup_matwm_training(config, device)

# 共有World Modelの作成と検証
shared_world_model, shared_wm_optimizer = MATWMAgent.create_shared_world_model(config, device)
dummy_agent = MATWMAgent(config, 'adversary_0', 0, device, shared_world_model=shared_world_model)

print('\n=== Weight Initialization ===')
initialize_matwm_weights(shared_world_model, dummy_agent.actor, dummy_agent.critic)

inspect_matwm_architecture(shared_world_model, dummy_agent.actor, dummy_agent.critic, config, device)

## 3. RND モジュールの設定

RND (Random Network Distillation) の設定:
- **obs_dim**: 観測次元（16次元にパディング済み）
- **hidden_dim**: 隠れ層の次元
- **feature_dim**: Target/Predictor の出力特徴次元
- **weight**: RND 内発的報酬の重み
- **normalize_observations**: 観測の正規化（Running Mean/Std）
- **normalize_rewards**: 報酬の正規化

In [ ]:
from curiosity_rnd_icm import RNDConfig, create_rnd_managers
import time

# Experiment configuration
EXPERIMENT_METHOD = 'rnd'
TIMESTAMP = time.strftime('%Y%m%d_%H%M%S')

# Output directories with method and timestamp
LOG_DIR = f'../../logs/{EXPERIMENT_METHOD}/{TIMESTAMP}'
RESULTS_DIR = f'../../results/{EXPERIMENT_METHOD}/{TIMESTAMP}'

rnd_config = RNDConfig(
    obs_dim=16,              # 観測次元（パディング後）
    hidden_dim=256,          # 隠れ層の次元
    feature_dim=128,         # Target/Predictor の出力次元
    weight=1.0,              # RND内発的報酬の重み
    normalize_observations=True,  # 観測の正規化
    normalize_rewards=True,       # 報酬の正規化
)

print('=== RND Configuration ===')
print(f'  Observation dim: {rnd_config.obs_dim}')
print(f'  Hidden dim: {rnd_config.hidden_dim}')
print(f'  Feature dim: {rnd_config.feature_dim}')
print(f'  Weight: {rnd_config.weight}')
print(f'  Normalize observations: {rnd_config.normalize_observations}')
print(f'  Normalize rewards: {rnd_config.normalize_rewards}')
print(f'  Log dir: {LOG_DIR}')

## 4. 訓練関数の定義

RND による内発的報酬を統合した訓練ループ:

1. Actor Network が行動を選択（warmup中はランダム）
2. 環境ステップを実行
3. RND が観測の新規性を評価し、内発的報酬を計算
4. `env_reward + intrinsic_reward` を replay buffer に格納
5. RND Predictor Network を定期的に訓練
6. World Model + Actor-Critic を訓練

In [ ]:
def train_matwm_with_rnd(config, rnd_config, save_dir='results', resume_from=None):
    """
    MATWM + RND 内発的報酬で訓練。

    Args:
        config: MATWMConfig
        rnd_config: RNDConfig
        save_dir: 保存先ディレクトリ
        resume_from: チェックポイントからの再開パス
    """
    # 環境作成
    env = make_env(max_cycles=config.max_cycles, seed=42)
    agent_names = env.agents

    # 共有 World Model
    shared_wm, shared_wm_opt = MATWMAgent.create_shared_world_model(config, device)
    print(f'Shared World Model: {sum(p.numel() for p in shared_wm.parameters())} params')

    # エージェント作成
    agents = {}
    for idx, name in enumerate(agent_names):
        agents[name] = MATWMAgent(config, name, idx, device, shared_world_model=shared_wm)

    # 重み初期化
    if resume_from is None:
        print('\n=== Initializing Weights ===')
        initialize_matwm_weights(shared_wm,
                                 list(agents.values())[0].actor,
                                 list(agents.values())[0].critic)
        for agent in agents.values():
            agent.actor.apply(init_weights)
            agent.critic.apply(init_weights)
        print('✓ Weight initialization complete')

    # RND マネージャ作成
    rnd_managers = create_rnd_managers(
        agent_names,
        rnd_config,
        device,
    )
    print(f'\n=== RND Managers Created ===')
    for name in agent_names:
        print(f'  {name}: Target={rnd_managers[name].target_net}, Predictor={rnd_managers[name].predictor_net}')

    # メトリクス
    episode_rewards = {name: [] for name in agent_names}
    episode_curiosity = {name: [] for name in agent_names}  # RND報酬の推移
    training_metrics = defaultdict(list)
    rnd_metrics = defaultdict(list)
    start_step = 0

    # チェックポイントからの再開
    if resume_from is not None and os.path.exists(resume_from):
        print(f'\n=== Resuming from: {resume_from} ===')
        episode_rewards, training_metrics, start_step = load_full_checkpoint(
            agents, shared_wm, shared_wm_opt, resume_from, device
        )
        print(f'✓ Resumed from step {start_step}')

    # 保存ディレクトリ
    os.makedirs(save_dir, exist_ok=True)
    timestamp = time.strftime('%Y_%m_%d_%H_%M_%S')
    run_dir = os.path.join(save_dir, f'matwm_rnd_{timestamp}')
    os.makedirs(run_dir, exist_ok=True)

    print(f'\n=== Starting MATWM + RND Training ===')
    print(f'Save directory: {run_dir}')
    print(f'Total steps: {config.total_steps}')
    print(f'Warmup steps: {config.warmup_steps}')
    print(f'RND weight: {rnd_config.weight}\n')

    # 訓練ループ
    global_step = start_step
    episode_count = 0
    min_data = config.wm_batch_length + 10
    pbar = tqdm(total=config.total_steps, initial=start_step, desc='Training')

    while global_step < config.total_steps:
        obs, info = env.reset()
        ep_reward = {name: 0.0 for name in agent_names}
        ep_intrinsic = {name: 0.0 for name in agent_names}

        for step in range(config.max_cycles):
            # 行動選択: Actor Network（warmup中はランダム）
            actions = {}
            for name, agent in agents.items():
                if global_step < config.warmup_steps:
                    actions[name] = env.action_space(name).sample()
                else:
                    actions[name] = agent.select_action(obs[name])

            # 環境ステップ
            next_obs, rewards, terms, truncs, infos = env.step(actions)
            done = {name: terms[name] or truncs[name] for name in agent_names}

            # RND 内発的報酬の計算と経験の格納
            for name, agent in agents.items():
                other_acts = {k: v for k, v in actions.items() if k != name}
                env_r = rewards[name]

                # RND 内発的報酬（観測のみで計算）
                intrinsic_r = 0.0
                if global_step >= min_data:
                    obs_padded = pad_observation(obs[name], config.max_obs_dim)
                    intrinsic_r = rnd_managers[name].compute_intrinsic_reward(obs_padded)

                total_r = env_r + intrinsic_r

                # Replay buffer に格納（env + RND報酬）
                agent.store_experience(
                    obs[name], actions[name], total_r,
                    next_obs[name], done[name], other_acts,
                )
                ep_reward[name] += env_r
                ep_intrinsic[name] += intrinsic_r

            obs = next_obs
            global_step += 1
            pbar.update(1)

            # World Model 訓練
            if global_step >= config.warmup_steps:
                wm_metrics = MATWMAgent.train_world_model_shared(
                    agents, config, device, shared_wm_opt
                )
                if wm_metrics:
                    for k, v in wm_metrics.items():
                        training_metrics[f'shared_{k}'].append(v)

                # Actor-Critic 訓練
                for name, agent in agents.items():
                    ac_metrics = agent.train_agent()
                    for k, v in ac_metrics.items():
                        training_metrics[f'{name}_{k}'].append(v)

                # RND Predictor 訓練（定期的に）
                if global_step % 10 == 0:  # 10ステップごと
                    for name, agent in agents.items():
                        # replay buffer から観測をサンプル
                        if len(agent.replay_buffer) >= config.wm_batch_size:
                            batch = agent.replay_buffer.sample(config.wm_batch_size)
                            obs_batch = np.array([pad_observation(exp[0], config.max_obs_dim) for exp in batch])
                            rnd_loss = rnd_managers[name].train(obs_batch)
                            rnd_metrics[f'{name}_rnd_loss'].append(rnd_loss)

            # ログ
            if global_step % config.log_interval == 0 and global_step >= config.warmup_steps:
                log_str = f'Step {global_step}: '
                for name in agent_names:
                    if episode_rewards[name]:
                        log_str += f'{name}={np.mean(episode_rewards[name][-10:]):.2f} '
                pbar.set_description(log_str)

            # チェックポイント保存
            if global_step % config.save_interval == 0 and global_step >= config.warmup_steps:
                ckpt_dir = os.path.join(run_dir, f'checkpoint_{global_step}')
                os.makedirs(ckpt_dir, exist_ok=True)
                for name, agent in agents.items():
                    agent.save(os.path.join(ckpt_dir, f'{name}.pt'))
                save_full_checkpoint(
                    agents, shared_wm, shared_wm_opt,
                    episode_rewards, training_metrics, global_step,
                    os.path.join(ckpt_dir, 'full_checkpoint.pt')
                )
                print(f'\n✓ Checkpoint saved at step {global_step}')

            if all(done.values()):
                break

        # エピソード終了
        for name in agent_names:
            episode_rewards[name].append(ep_reward[name])
            episode_curiosity[name].append(ep_intrinsic[name])

        episode_count += 1

    pbar.close()
    env.close()

    # 最終チェックポイント
    final_dir = os.path.join(run_dir, 'final')
    os.makedirs(final_dir, exist_ok=True)
    for name, agent in agents.items():
        agent.save(os.path.join(final_dir, f'{name}.pt'))
    save_full_checkpoint(
        agents, shared_wm, shared_wm_opt,
        episode_rewards, training_metrics, global_step,
        os.path.join(final_dir, 'full_checkpoint.pt')
    )

    print(f'\n=== Training Complete ===')
    print(f'Total episodes: {episode_count}')
    print(f'Final checkpoint: {final_dir}')
    for name in agent_names:
        if episode_rewards[name]:
            r = episode_rewards[name][-100:] if len(episode_rewards[name]) >= 100 else episode_rewards[name]
            print(f'  {name}: mean reward = {np.mean(r):.2f}')

    # RND統計
    print('\n=== RND Statistics ===')
    for name in agent_names:
        if episode_curiosity[name]:
            c = episode_curiosity[name]
            print(f'  {name}: mean intrinsic reward = {np.mean(c):.2f}')

    return agents, episode_rewards, training_metrics, episode_curiosity, rnd_metrics

print('Training function defined. ✓')

## 5. 訓練の実行

**注意**: 完全な訓練には時間がかかります（GPUで数時間）。
短時間テストの場合は `config.total_steps` を小さくしてください。

In [ ]:
# 訓練実行
print('=' * 70)
print('MATWM + RND Training')
print('=' * 70)

agents, episode_rewards, training_metrics, episode_curiosity, rnd_metrics = \
    train_matwm_with_rnd(config, rnd_config, save_dir=RESULTS_DIR)

# チェックポイントからの再開（コメントアウト解除して使用）
# checkpoint_path = '../../results/rnd/YYYYMMDD_HHMMSS/checkpoint_25000/full_checkpoint.pt'
# agents, episode_rewards, training_metrics, episode_curiosity, rnd_metrics = \
#     train_matwm_with_rnd(config, rnd_config, save_dir=RESULTS_DIR, resume_from=checkpoint_path)

## 6. エージェントの評価

In [ ]:
def evaluate_agents(agents, num_episodes=20):
    """訓練済みエージェントの性能を評価"""
    env = make_env(max_cycles=config.max_cycles)
    agent_names = list(agents.keys())
    eval_rewards = {name: [] for name in agent_names}

    for ep in range(num_episodes):
        obs, _ = env.reset()
        ep_reward = {name: 0.0 for name in agent_names}

        for step in range(config.max_cycles):
            actions = {name: agent.select_action(obs[name], deterministic=True)
                       for name, agent in agents.items()}
            next_obs, rewards, terms, truncs, _ = env.step(actions)

            for name in agent_names:
                ep_reward[name] += rewards[name]

            obs = next_obs
            if all(terms[n] or truncs[n] for n in agent_names):
                break

        for name in agent_names:
            eval_rewards[name].append(ep_reward[name])

        print(f'Episode {ep+1}/{num_episodes}: ' +
              ' '.join(f'{n}={ep_reward[n]:.2f}' for n in agent_names))

    env.close()

    print('\n=== Evaluation Results ===')
    for name in agent_names:
        print(f'  {name}: Mean={np.mean(eval_rewards[name]):.2f}, Std={np.std(eval_rewards[name]):.2f}')

    return eval_rewards

print('Evaluating trained agents...')
eval_rewards = evaluate_agents(agents, num_episodes=20)

## 7. 訓練結果の可視化

In [ ]:
# 詳細可視化
print('=' * 70)
print('TRAINING VISUALIZATION')
print('=' * 70)
plot_training_progress(episode_rewards, training_metrics, save_path='results/training_curves_rnd.png')

In [ ]:
# 学習曲線 + RND報酬の推移
agent_names = list(episode_rewards.keys())

fig, axes = plt.subplots(3, 2, figsize=(16, 14))

# (0,0) Episode rewards
ax = axes[0, 0]
for name in agent_names:
    r = episode_rewards[name]
    if len(r) > 0:
        w = min(10, len(r))
        if len(r) >= w:
            ma = np.convolve(r, np.ones(w)/w, mode='valid')
            ax.plot(ma, label=name)
ax.set_title('Episode Rewards (Moving Average)')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# (0,1) Shared WM Total Loss
ax = axes[0, 1]
key = 'shared_wm_total_loss'
if key in training_metrics and training_metrics[key]:
    ax.plot(training_metrics[key], alpha=0.7)
ax.set_title('Shared World Model Loss')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)

# (1,0) Actor Loss
ax = axes[1, 0]
for name in agent_names:
    key = f'{name}_actor_loss'
    if key in training_metrics and training_metrics[key]:
        ax.plot(training_metrics[key], label=name, alpha=0.7)
ax.set_title('Actor Loss')
ax.set_xlabel('Training Step')
ax.legend()
ax.grid(True, alpha=0.3)

# (1,1) RND Predictor Loss
ax = axes[1, 1]
for name in agent_names:
    key = f'{name}_rnd_loss'
    if key in rnd_metrics and rnd_metrics[key]:
        ax.plot(rnd_metrics[key], label=name, alpha=0.7)
ax.set_title('RND Predictor Loss')
ax.set_xlabel('Training Step')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# (2,0) RND Intrinsic Rewards per Episode
ax = axes[2, 0]
for name in agent_names:
    c = episode_curiosity.get(name, [])
    if c:
        ax.plot(c, label=name, alpha=0.7)
ax.set_title('RND Intrinsic Reward per Episode')
ax.set_xlabel('Episode')
ax.set_ylabel('Intrinsic Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# (2,1) Total Reward (Env + RND) per Episode
ax = axes[2, 1]
for name in agent_names:
    env_r = episode_rewards.get(name, [])
    rnd_r = episode_curiosity.get(name, [])
    if env_r and rnd_r:
        total_r = [e + r for e, r in zip(env_r, rnd_r)]
        ax.plot(total_r, label=name, alpha=0.7)
ax.set_title('Total Reward (Env + RND) per Episode')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_curves_rnd_detailed.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/training_curves_rnd_detailed.png')

## 8. まとめと考察

### 実装した内容

本 Notebook では、MATWM に **RND (Random Network Distillation) による内発的報酬** を統合した。

#### 主要コンポーネント

1. **World Model**（MATWM 論文通り）
   - Encoder/Decoder (Categorical VAE)
   - Dynamics Model (Transformer)
   - Reward/Continuation Predictor
   - Teammate Predictor

2. **RND (Random Network Distillation)**（本研究の実装）
   - **Target Network**: ランダム初期化・固定
   - **Predictor Network**: 訓練可能
   - **内発的報酬**: 予測誤差 = 観測の新規性

3. **Actor-Critic**（MATWM 論文通り）
   - Actor: 想像軌道上のポリシー学習
   - Critic: 価値関数の推定

---

### RND の利点と限界

#### 利点

1. **シンプル**: 環境ダイナミクスのモデル化不要
2. **スケーラブル**: 高次元観測空間でも機能
3. **安定**: カウントベース手法より安定した報酬
4. **汎用性**: どんな強化学習アルゴリズムにも適用可能

#### 限界

1. **確率的環境に弱い**:
   - 確率的ダイナミクスがある環境では、同じ状態でも常に予測誤差が大きくなる
   - TV のノイズ画面で高報酬を得続ける問題（noisy-TV problem）

2. **行動の影響を考慮しない**:
   - RND は観測のみを評価し、「自分の行動が環境に与える影響」を測定しない
   - ICM (Intrinsic Curiosity Module) などの forward model ベース手法の方が効果的な場合がある

3. **マルチエージェント特有の課題**:
   - 他エージェントの行動による状態変化も「新規」と判断される
   - Social Curiosity（TeammatePredictor の予測誤差）の方が協調学習には適している可能性

---

### 今後の改善案

1. **ICM (Intrinsic Curiosity Module)**: Forward model で「自分の行動の影響」を測定
2. **NGU (Never Give Up)**: Episodic novelty + Lifelong novelty の二重構造
3. **Social Curiosity との統合**: RND + TeammatePredictor の予測誤差
4. **LLM 意味的好奇心**: 戦略的新規性の評価

---

### 参考文献

- Burda et al. (2018). "Exploration by Random Network Distillation." ICLR 2019.
- Pathak et al. (2017). "Curiosity-driven Exploration by Self-supervised Prediction." ICML 2017.
- Badia et al. (2020). "Never Give Up: Learning Directed Exploration Strategies." ICLR 2020.